# NYC Airbnb Listings — Data Cleaning, EDA & Price Prediction

**Dataset:** [Google Drive](https://drive.google.com/file/d/1Ro_jQ6MqHwT4UC2Ygq3_SpWKtZ9JcYlX/view?usp=sharing)

This notebook downloads the NYC Airbnb listings dataset (~49k rows), cleans it, explores patterns, and predicts **nightly price** using:
1. **Random Forest** (classical ML baseline)
2. **Feedforward neural network** (TensorFlow/Keras)

**Target:** `price` (regression)  
**Runtime:** ~5–10 minutes on Colab CPU (GPU optional for the NN).


In [ ]:
# Install gdown for Google Drive download (Colab)
!pip install -q gdown seaborn plotly


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import gdown

sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams["figure.figsize"] = (10, 5)


## 1. Download & load data


In [ ]:
FILE_ID = "1Ro_jQ6MqHwT4UC2Ygq3_SpWKtZ9JcYlX"
url = f"https://drive.google.com/uc?id={FILE_ID}"
gdown.download(url, "nyc_airbnb.csv", quiet=False)

df_raw = pd.read_csv("nyc_airbnb.csv")
print(f"Shape: {df_raw.shape}")
df_raw.head()


## 2. Data cleaning


In [ ]:
df = df_raw.copy()

# Parse dates
df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")

# Drop rows with missing review activity (same rows missing last_review)
before = len(df)
df = df.dropna(subset=["reviews_per_month"])
print(f"Dropped {before - len(df):,} rows with missing reviews_per_month")

# Trim obvious listing errors
df = df[df["price"] > 0]
df = df[df["price"] <= 500]          # cap extreme luxury outliers for stable modeling
df = df[df["minimum_nights"] <= 365]

# Fill remaining categorical nulls
df["host_name"] = df["host_name"].fillna("Unknown")
df["name"] = df["name"].fillna("Unnamed listing")

# Remove duplicate listing ids if any
df = df.drop_duplicates(subset=["id"])

print(f"Cleaned shape: {df.shape}")
print("\nMissing values after cleaning:")
print(df.isnull().sum().sort_values(ascending=False).head())


## 3. Exploratory data analysis


In [ ]:
df.describe(include="all").T


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x="neighbourhood_group", order=df["neighbourhood_group"].value_counts().index, ax=axes[0])
axes[0].set_title("Listings by borough")
axes[0].tick_params(axis="x", rotation=20)

sns.boxplot(data=df, x="room_type", y="price", ax=axes[1])
axes[1].set_title("Price distribution by room type")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["price"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("Nightly price distribution (cleaned)")

sns.scatterplot(data=df.sample(min(5000, len(df)), random_state=42),
                x="number_of_reviews", y="price", hue="room_type", alpha=0.4, ax=axes[1])
axes[1].set_title("Reviews vs price (sample)")
plt.tight_layout()
plt.show()


In [ ]:
# Geographic scatter — sample for performance
sample = df.sample(min(8000, len(df)), random_state=42)
fig = px.scatter(
    sample, x="longitude", y="latitude", color="neighbourhood_group",
    size="price", hover_data=["room_type", "price"],
    title="NYC Airbnb listings (sampled)", opacity=0.6,
    height=550
)
fig.show()


In [ ]:
numeric_cols = [
    "price", "minimum_nights", "number_of_reviews", "reviews_per_month",
    "calculated_host_listings_count", "availability_365", "latitude", "longitude"
]
corr = df[numeric_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix (numeric features)")
plt.tight_layout()
plt.show()


## 4. Modeling — predict nightly `price`

We use location, room type, and host/listing activity features.  
80/20 train/test split, stratified by borough.


In [ ]:
FEATURES = [
    "neighbourhood_group", "neighbourhood", "room_type",
    "latitude", "longitude", "minimum_nights", "number_of_reviews",
    "reviews_per_month", "calculated_host_listings_count", "availability_365",
]
TARGET = "price"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=X["neighbourhood_group"]
)

cat_features = ["neighbourhood_group", "neighbourhood", "room_type"]
num_features = [c for c in FEATURES if c not in cat_features]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ("num", StandardScaler(), num_features),
])

def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    print(f"{name}")
    print(f"  MAE:  ${mae:.2f}")
    print(f"  RMSE: ${rmse:.2f}")
    print(f"  R²:   {r2:.3f}")
    return {"model": name, "mae": mae, "rmse": rmse, "r2": r2}


### 4.1 Random Forest (baseline)


In [ ]:
rf_pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200, max_depth=24, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    )),
])

rf_pipe.fit(X_train, y_train)
rf_pred = rf_pipe.predict(X_test)
rf_metrics = evaluate(y_test, rf_pred, "Random Forest")


In [ ]:
# Feature importance (top categories from forest)
importances = rf_pipe.named_steps["model"].feature_importances_
# Get feature names after one-hot
ohe = rf_pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(cat_features)
all_names = list(cat_names) + num_features
imp_df = pd.DataFrame({"feature": all_names, "importance": importances}).sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(10, 5))
sns.barplot(data=imp_df, y="feature", x="importance")
plt.title("Top 15 Random Forest feature importances")
plt.tight_layout()
plt.show()


### 4.2 Feedforward neural network (deep learning)


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Prepare numeric matrix for Keras (same split)
preprocess_nn = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_features),
    ("num", StandardScaler(), num_features),
])

X_train_enc = preprocess_nn.fit_transform(X_train)
X_test_enc = preprocess_nn.transform(X_test)

# Convert sparse to dense for Keras
if hasattr(X_train_enc, "toarray"):
    X_train_enc = X_train_enc.toarray()
    X_test_enc = X_test_enc.toarray()

y_train_scaled = y_train.values.astype("float32")
y_test_scaled = y_test.values.astype("float32")

tf.random.set_seed(42)

model = keras.Sequential([
    layers.Input(shape=(X_train_enc.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(64, activation="relu"),
    layers.Dense(1),
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])

history = model.fit(
    X_train_enc, y_train_scaled,
    validation_split=0.15,
    epochs=40,
    batch_size=256,
    verbose=1,
)

nn_pred = model.predict(X_test_enc, verbose=0).flatten()
nn_metrics = evaluate(y_test_scaled, nn_pred, "Neural Network (MLP)")


In [ ]:
# Training curves
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist["loss"], label="train")
axes[0].plot(hist["val_loss"], label="val")
axes[0].set_title("MSE loss"); axes[0].legend()
axes[1].plot(hist["mae"], label="train")
axes[1].plot(hist["val_mae"], label="val")
axes[1].set_title("MAE"); axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# Predicted vs actual (Random Forest)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sample_idx = np.random.choice(len(y_test), size=min(3000, len(y_test)), replace=False)

axes[0].scatter(y_test.iloc[sample_idx], rf_pred[sample_idx], alpha=0.3, s=10)
axes[0].plot([25, 500], [25, 500], "r--", lw=2)
axes[0].set_xlabel("Actual price"); axes[0].set_ylabel("Predicted")
axes[0].set_title("Random Forest: predicted vs actual")

axes[1].scatter(y_test.iloc[sample_idx], nn_pred[sample_idx], alpha=0.3, s=10, color="teal")
axes[1].plot([25, 500], [25, 500], "r--", lw=2)
axes[1].set_xlabel("Actual price"); axes[1].set_ylabel("Predicted")
axes[1].set_title("Neural Network: predicted vs actual")
plt.tight_layout()
plt.show()


## 5. Model comparison & conclusions


In [ ]:
results = pd.DataFrame([rf_metrics, nn_metrics]).set_index("model")
results


### Key findings

- **Manhattan** has the most listings; **Entire home/apt** dominates inventory but **Private room** is common in outer boroughs.
- **Price** is right-skewed; `room_type` and **borough/neighbourhood** are strong drivers.
- **Random Forest** usually wins or ties on tabular Airbnb-style data thanks to mixed categorical + numeric features.
- The **MLP** captures nonlinear patterns but needs tuning (depth, learning rate, more epochs) to match tree ensembles.

### Possible extensions
- Log-transform `price` for RMSE stability
- Text embeddings on listing `name` for luxury/budget signals
- Time-based split using `last_review` instead of random split
- Gradient boosting (XGBoost/LightGBM) for production-grade accuracy
